In [3]:
import sys

sys.path.append("../nlp")

In [6]:
import pandas as pd
from nlp_detector import NLPFraudDetector

In [8]:
# LOAD NLP FRAUD DETECTOR
detector = NLPFraudDetector()

print("Detector Loaded Successfully")

Detector Loaded Successfully


In [9]:
# LOAD CLEANED DATASET

df = pd.read_csv(
    "../data/cleaned_data.csv"
)

print(df.shape)

(4661, 29)


In [10]:
# VIEW ALL COLUMNS


df.columns.tolist()

['title',
 'company',
 'location_clean',
 'stipend_clean',
 'duration_clean',
 'start_date',
 'apply_by',
 'openings_clean',
 'employment_type',
 'work_mode',
 'skills',
 'description',
 'perks',
 'job_url',
 'company_website',
 'recruiter_email',
 'domain_name',
 'platform',
 'scraped_date',
 'salary_clean',
 'experience_required_clean',
 'posted_days_ago',
 'company_about',
 'is_internship',
 'has_description',
 'has_skills',
 'has_company_website',
 'has_recruiter_email',
 'missing_info_score']

In [11]:
# COMBINE IMPORTANT TEXT COLUMNS

df["combined_text"] = (

    df["title"].fillna("")

    + " "

    + df["description"].fillna("")

    + " "

    + df["skills"].fillna("")

    + " "

    + df["company_about"].fillna("")

)

In [12]:
# SAMPLE COMBINED TEXT

print(

    df["combined_text"].iloc[0][:1000]

)

finance research analyst - internship wfh & part time selected intern's day-to-day responsibilities include: 1. assist in creating detailed reports for buy/hold/sell recommendations. 2. help in tracking portfolio performance. 3. collect, analyze, and interpret financial data from companies, sectors, and industries. 4. track stock markets, economic indicators, and global financial news. attention to detail, time management, critical thinking, analytical thinking, problem solving, english proficiency (spoken), stock trading 


In [13]:
# TEST NLP ON FIRST RECORD

result = detector.analyze(

    df["combined_text"].iloc[0]

)

result

{'clean_text': 'finance research analyst internship wfh part time selected intern s day to day responsibilities include assist in creating detailed reports for buy hold sell recommendations help in tracking portfolio performance collect analyze and interpret financial data from companies sectors and industries track stock markets economic indicators and global financial news attention to detail time management critical thinking analytical thinking problem solving english proficiency spoken stock trading',
 'lemmatized_text': 'finance research analyst internship wfh time select intern s day day responsibility include assist create detailed report buy hold sell recommendation help track portfolio performance collect analyze interpret financial datum company sector industry track stock market economic indicator global financial news attention detail time management critical thinking analytical thinking problem solve english proficiency speak stock trading',
 'description_length': 66,
 're

In [ ]:
# GENERATE NLP FEATURES

results = []

for index, text in enumerate(

    df["combined_text"]

):

    result = detector.analyze(text)

    results.append(result)

    if index % 500 == 0:

        print(

            f"Processed {index} jobs"

        )

print("Completed")

Processed 0 jobs
Processed 500 jobs
Processed 1000 jobs
Processed 1500 jobs
Processed 2000 jobs
Processed 2500 jobs
Processed 3000 jobs
Processed 3500 jobs
Processed 4000 jobs
Processed 4500 jobs
Completed


In [ ]:
# NLP FEATURES DATAFRAME

features_df = pd.DataFrame(

    results

)

features_df.head()

,clean_text,lemmatized_text,description_length,readability_score,keyword_score,matched_keywords,skill_count,found_skills,urgency_score,contact_score,has_description,missing_info_score,scam_density,risk_score,prediction
0,finance research analyst internship wfh part t...,finance research analyst internship wfh time s...,66,-39.61,0,[],0,[],0,0,1,0,0.000,0,GENUINE
1,digital marketing female selected intern s day...,digital marketing female select intern s day d...,36,-43.55,1,[whatsapp],0,[],0,1,1,0,0.028,25,GENUINE
2,social media marketing internship social media...,social medium marketing internship social medi...,507,-492.32,0,[],0,[],0,0,1,0,0.000,0,GENUINE
3,pre sales internship selected intern s day to ...,pre sale internship select intern s day day re...,93,-83.14,0,[],0,[],0,0,1,0,0.000,0,GENUINE
4,marketing support female candidates internship...,marketing support female candidate internship ...,207,-150.81,1,[whatsapp],1,[excel],0,1,1,0,0.005,25,GENUINE


In [16]:
# RISK SCORE DISTRIBUTION

features_df["risk_score"].describe()

count    4661.000000
mean        2.645355
std         6.510684
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        50.000000
Name: risk_score, dtype: float64

In [17]:
# TOP RISK JOBS

features_df.sort_values(

    by="risk_score",

    ascending=False

).head(20)

,clean_text,lemmatized_text,description_length,readability_score,keyword_score,matched_keywords,skill_count,found_skills,urgency_score,contact_score,has_description,missing_info_score,scam_density,risk_score,prediction
1446,telecalling internship selected intern s day t...,telecalle internship select intern s day day r...,146,-125.62,2,"[telegram, whatsapp]",1,[excel],0,2,1,0,0.014,50,SCAM
2698,community management internship wfh part time ...,community management internship wfh time commu...,184,-160.16,2,"[telegram, whatsapp]",1,[excel],0,2,1,0,0.011,50,SCAM
2805,campus ambassador internship wfh part time sel...,campus ambassador internship wfh time select i...,175,-136.12,2,"[telegram, whatsapp]",0,[],0,2,1,0,0.011,50,SCAM
3930,sr customer service executive mega drive hirin...,sr customer service executive mega drive hire ...,179,-136.02,2,"[whatsapp, limited seats]",0,[],2,1,1,0,0.011,50,SCAM
4270,customer support greetings from hucon solution...,customer support greeting hucon solution huge ...,141,-82.68,2,"[whatsapp, immediate joining]",0,[],1,1,1,0,0.014,45,SCAM
4443,accountant company tharwani infrastructure loc...,accountant company tharwani infrastructure loc...,62,-25.29,2,"[whatsapp, immediate joining]",0,[],1,1,1,0,0.032,45,SCAM
4288,international customer support voice process m...,international customer support voice process m...,94,-88.38,2,"[whatsapp, immediate joining]",1,[excel],1,1,1,0,0.021,45,SCAM
3988,urgent hiring freshers architecture hiring fre...,urgent hire fresher architecture hire fresher ...,143,-113.43,1,[whatsapp],1,[excel],2,1,1,0,0.007,35,GENUINE
3208,telecaller bpo call center hsc fresher custome...,telecaller bpo center hsc fresher customer car...,19,27.26,1,[immediate joining],0,[],2,0,1,1,0.053,35,GENUINE
3868,lpa career counsellor noida urgent hiring care...,lpa career counsellor noida urgent hire career...,81,-66.51,1,[whatsapp],0,[],2,1,1,0,0.012,35,GENUINE


In [ ]:
# MERGE FEATURES WITH DATASET

final_df = pd.concat(

    [

        df.reset_index(drop=True),

        features_df.reset_index(drop=True)

    ],

    axis=1

)

final_df.head()

,title,company,location_clean,stipend_clean,duration_clean,start_date,apply_by,openings_clean,employment_type,work_mode,...,matched_keywords,skill_count,found_skills,urgency_score,contact_score,has_description,missing_info_score,scam_density,risk_score,prediction
0,finance research analyst - internship wfh & pa...,edify equity,delhi,rs 1000 - 1100 per month,1 month,immediately,2026-06-28 00:00:00,20.0,part time,remote,...,[],0,[],0,0,1,0,0.000,0,GENUINE
1,digital marketing female,wear it like this,"secunderabad, hyderabad",rs 10000 - 15000 per month,3 month,immediately,2026-06-27 00:00:00,NaN,full time,onsite,...,[whatsapp],0,[],0,1,1,0,0.028,25,GENUINE
2,social media marketing - internship,madhuram hospitality,bangalore,rs 5000 - 10000 per month,3 month,immediately,2026-06-28 00:00:00,2.0,full time,hybrid,...,[],0,[],0,0,1,0,0.000,0,GENUINE
3,pre sales - internship,dronacharya group,bhilwara,rs 9500 - 16000 per month,3 month,immediately,2026-06-25 00:00:00,6.0,full time,onsite,...,[],0,[],0,0,1,0,0.000,0,GENUINE
4,marketing support female candidates - internship,neeti brand accelerator,mumbai,rs 11000 - 17000 per month,3 month,immediately,2026-06-28 00:00:00,2.0,full time,onsite,...,[whatsapp],1,[excel],0,1,1,0,0.005,25,GENUINE


In [19]:
# FINAL DATASET SHAPE

final_df.shape

(4661, 45)

In [ ]:
# SAVE NLP FEATURE DATASET

final_df.to_csv(

    "../data/nlp_features.csv",

    index=False

)

print(

    "Saved: nlp_features.csv"

)

Saved: nlp_features.csv


In [21]:
# FRAUD PREDICTION COUNT

features_df["prediction"].value_counts()

prediction
GENUINE    4654
SCAM          7
Name: count, dtype: int64

,title,company,risk_score,matched_keywords,prediction
1446,telecalling - internship,search digitally,50,"[telegram, whatsapp]",SCAM
2698,community management - internship wfh & part time,nayepankh foundation,50,"[telegram, whatsapp]",SCAM
2805,campus ambassador - internship wfh & part time,base,50,"[telegram, whatsapp]",SCAM
3930,sr customer service executive,vibrantzz management services hiring for inter...,50,"[whatsapp, limited seats]",SCAM
4270,customer support,hucon solutions india pvt.ltd.,45,"[whatsapp, immediate joining]",SCAM
4443,accountant,tharwani infrastructure,45,"[whatsapp, immediate joining]",SCAM
4288,international customer support voice process m...,tech mahindra ltd.,45,"[whatsapp, immediate joining]",SCAM
3988,urgent hiring freshers architecture,millicon consultant engineers pvt ltd,35,[whatsapp],GENUINE
3208,telecaller bpo call center hsc fresher custome...,altruist technologies,35,[immediate joining],GENUINE
3868,9lpa career counsellor noida 125,sharda consultancy services,35,[whatsapp],GENUINE


risk_score
0     3679
10     403
5      325
25     174
30      45
20      12
15       8
35       8
50       4
45       3
Name: count, dtype: int64